# 📄 Contract Clause Risk Analyzer
> Rule-based NLP system using NLTK + Regex — No LLMs, No APIs

**What this notebook does:**
1. Installs & imports dependencies
2. Defines the full analyzer (preprocessor → scorer → explainer)
3. Runs sample clauses and prints JSON results
4. Shows a multi-clause contract analysis

In [28]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install nltk --quiet

In [29]:
# ── Step 2: Paste the full analyzer code ─────────────────────────────────────
# (In Colab, either paste analyzer.py contents here OR upload the file and run:)
# from analyzer import ContractRiskAnalyzer

# ─── For self-contained Colab use, the full source is included below ─────────

import re
import json
import nltk
from collections import defaultdict

nltk.download('punkt',      quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('punkt_tab',  quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus   import stopwords

print('NLTK ready')

NLTK ready


In [30]:
# ── Step 3: Risk keyword dictionary ──────────────────────────────────────────

RISK_KEYWORDS = {
    'indemnify':             {'weight': 7,  'category': 'Liability',         'explanation': 'Requires one party to compensate the other for losses.'},
    'indemnification':       {'weight': 7,  'category': 'Liability',         'explanation': 'A clause transferring liability — risky if uncapped.'},
    'hold harmless':         {'weight': 8,  'category': 'Liability',         'explanation': 'Waives right to sue — may eliminate critical legal remedies.'},
    'unlimited liability':   {'weight': 10, 'category': 'Liability',         'explanation': 'Exposes a party to uncapped financial loss.'},
    'terminate':             {'weight': 5,  'category': 'Termination',       'explanation': 'Grants termination rights — risk depends on conditions.'},
    'termination for convenience': {'weight': 8, 'category': 'Termination', 'explanation': 'Allows termination without cause — no contractual stability.'},
    'without cause':         {'weight': 7,  'category': 'Termination',       'explanation': 'Enables dismissal with no reason — highly unfavorable.'},
    'immediate termination': {'weight': 9,  'category': 'Termination',       'explanation': 'Allows instant contract end with no notice period.'},
    'notice period':         {'weight': 3,  'category': 'Termination',       'explanation': 'Specifies advance notice required before termination.'},
    'intellectual property': {'weight': 6,  'category': 'IP Rights',         'explanation': 'Governs ownership of created work.'},
    'work for hire':         {'weight': 7,  'category': 'IP Rights',         'explanation': 'Classifies work as employer-owned — creator loses all rights.'},
    'assign':                {'weight': 5,  'category': 'IP Rights',         'explanation': 'Transfers rights to another party — often irrevocable.'},
    'proprietary':           {'weight': 4,  'category': 'IP Rights',         'explanation': 'Marks info as exclusively owned — mishandling triggers liability.'},
    'exclusive license':     {'weight': 6,  'category': 'IP Rights',         'explanation': 'Grants rights to only one party — restricts licensor.'},
    'confidential':          {'weight': 4,  'category': 'Confidentiality',   'explanation': 'Marks information as protected.'},
    'non-disclosure':        {'weight': 5,  'category': 'Confidentiality',   'explanation': 'Prohibits sharing specified information.'},
    'nda':                   {'weight': 5,  'category': 'Confidentiality',   'explanation': 'NDA reference — check scope, duration, and exclusions.'},
    'trade secret':          {'weight': 7,  'category': 'Confidentiality',   'explanation': 'Legally protected proprietary info — disclosure carries penalties.'},
    'arbitration':           {'weight': 6,  'category': 'Dispute Resolution','explanation': 'Waives right to jury trial — limits appeal rights.'},
    'governing law':         {'weight': 4,  'category': 'Dispute Resolution','explanation': 'Sets which jurisdiction\'s law applies.'},
    'jurisdiction':          {'weight': 4,  'category': 'Dispute Resolution','explanation': 'Defines where disputes must be resolved.'},
    'class action waiver':   {'weight': 8,  'category': 'Dispute Resolution','explanation': 'Prevents joining group lawsuits.'},
    'penalty':               {'weight': 7,  'category': 'Financial',         'explanation': 'Specifies financial punishment for breach.'},
    'liquidated damages':    {'weight': 6,  'category': 'Financial',         'explanation': 'Pre-agreed breach compensation — may be disproportionate.'},
    'late payment':          {'weight': 5,  'category': 'Financial',         'explanation': 'Triggers interest or penalties for delayed payment.'},
    'auto-renewal':          {'weight': 6,  'category': 'Financial',         'explanation': 'Contract renews automatically — missing opt-out window is costly.'},
    'price increase':        {'weight': 5,  'category': 'Financial',         'explanation': 'Permits unilateral price changes.'},
    'force majeure':         {'weight': 4,  'category': 'Force Majeure',     'explanation': 'Excuses performance during unforeseen events.'},
    'limitation of liability': {'weight': 7,'category': 'Liability',         'explanation': 'Caps what can be recovered — may prevent full compensation.'},
    'waiver':                {'weight': 5,  'category': 'Rights',            'explanation': 'Gives up a legal right — often difficult to reverse.'},
    'sole remedy':           {'weight': 7,  'category': 'Rights',            'explanation': 'Restricts available legal remedies to one option only.'},
    'non-compete':           {'weight': 7,  'category': 'Restrictions',      'explanation': 'Restricts working for competitors.'},
    'non-solicitation':      {'weight': 6,  'category': 'Restrictions',      'explanation': 'Prevents poaching clients or employees.'},
    'restraint of trade':    {'weight': 8,  'category': 'Restrictions',      'explanation': 'Broadly limits business activities — courts scrutinize these.'},
}

print(f'Loaded {len(RISK_KEYWORDS)} risk keywords')

Loaded 34 risk keywords


In [31]:
RECOMMENDATIONS = {
    "Automatic Renewal":
        "Review cancellation notice requirements before renewal.",

    "Indemnification Clause":
        "Consider negotiating mutual indemnification provisions.",

    "Hold Harmless":
        "Review liability allocation between parties.",

    "Penalty Clause":
        "Ensure penalties are proportionate and capped.",

    "Liquidated Damages":
        "Verify damages are reasonable and legally enforceable.",

    "Termination Without Notice":
        "Request advance notice before termination.",

    "Sole Discretion":
        "Seek objective criteria instead of unilateral discretion.",

    "Limitation of Liability":
        "Review whether liability limits are appropriate.",

    "Binding Arbitration":
        "Evaluate dispute resolution alternatives.",

    "Modification Without Consent":
        "Require mutual agreement for contract changes."
}

In [32]:
def generate_recommendations(matches):

    recommendations = []

    for m in matches:

        keyword = m["keyword"]

        if keyword == "automatic renewal":
            recommendations.append(
                "Review cancellation notice requirements before renewal."
            )

        elif keyword == "indemnify":
            recommendations.append(
                "Consider negotiating mutual indemnification provisions."
            )

        elif keyword == "hold harmless":
            recommendations.append(
                "Review liability allocation between parties."
            )

        elif keyword == "penalty":
            recommendations.append(
                "Ensure penalties are proportionate and capped."
            )

        elif keyword == "liquidated damages":
            recommendations.append(
                "Verify damages are reasonable and enforceable."
            )

    return list(set(recommendations))

In [33]:
# ── Step 4: Core classes ──────────────────────────────────────────────────────

class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        legal_preserve  = {'not','no','nor','without','only','sole','all','any','none','never','shall','will','must','may','cannot'}
        self.stop_words -= legal_preserve

    def preprocess(self, text):
        lowered = text.lower()
        cleaned = re.sub(r'\s+', ' ', lowered)
        cleaned = re.sub(r"[^a-z0-9\s\-'.]", ' ', cleaned).strip()
        tokens  = word_tokenize(cleaned)
        filtered = [t for t in tokens if t not in self.stop_words and len(t) > 1]
        return {'original': text, 'cleaned': cleaned, 'tokens': tokens, 'filtered_tokens': filtered}


class RiskScorer:
    THRESHOLDS = {
    'Low': (0, 4),
    'Medium': (5, 14),
    'High': (15, float('inf'))
}

    def __init__(self):
        self.single_words = {k: v for k, v in RISK_KEYWORDS.items() if ' ' not in k and '-' not in k}
        self.phrases      = {k: v for k, v in RISK_KEYWORDS.items() if ' ' in k or '-' in k}

    def score_clause(self, preprocessed):
        tokens, cleaned = preprocessed['filtered_tokens'], preprocessed['cleaned']
        seen = set()
        matches = []

        # Phrase match first
        for phrase, info in self.phrases.items():
            pattern = r'\b' + r'\s+'.join(re.escape(w) for w in phrase.split()) + r'\b'
            if re.search(pattern, cleaned) and phrase not in seen:
                seen.add(phrase)
                matches.append({'keyword': phrase, **info})

        # Single-word match
        for token in tokens:
            if token in self.single_words and token not in seen:
                seen.add(token)
                matches.append({'keyword': token, **self.single_words[token]})

        score = sum(m['weight'] for m in matches)
        risk  = next(tier for tier, (lo, hi) in self.THRESHOLDS.items() if lo <= score <= hi)
        categories = defaultdict(list)
        for m in matches:
            categories[m['category']].append(m['keyword'])

        return {'matches': matches, 'total_score': score, 'risk_level': risk, 'categories': dict(categories)}


class ContractRiskAnalyzer:
    SUMMARIES = {
        'Low':    'Minor risk indicators. Review recommended.',
        'Medium': 'Notable risk factors. Legal review advisable.',
        'High':   'Significant legal risk. Do NOT sign without expert review.'
    }

    def __init__(self):
        self.pre    = TextPreprocessor()
        self.scorer = RiskScorer()

    def analyze(self, clause, clause_id='clause_1'):
        prep    = self.pre.preprocess(clause)
        scoring = self.scorer.score_clause(prep)
        recommendations = generate_recommendations(scoring["matches"])
        flags   = sorted(
            [{'flagged_term': m['keyword'], 'risk_category': m['category'],
              'risk_weight': f"{m['weight']}/10", 'reason': m['explanation']}
             for m in scoring['matches']],
            key=lambda x: int(x['risk_weight'].split('/')[0]), reverse=True
        )
        category_summary = {category: len(keywords)for category, keywords in scoring ['categories'].items()}
        return {
            'clause_id':     clause_id,
            'original_text': clause,
            'risk_level':    scoring['risk_level'],
            'risk_score':    scoring['total_score'],
            'summary': (
                f"{self.SUMMARIES[scoring['risk_level']]} "
                f"Detected {len(scoring['matches'])} risk indicators "
                f"with a total score of {scoring['total_score']}."
            ),
            'flags':         flags,
            'categories':    scoring['categories'],
            'category_breakdown': category_summary,
            'recommendations': recommendations
        }


    def analyze_contract(self, clauses):
      results = []

      for i, c in enumerate(clauses):

          if isinstance(c, str):
              result = self.analyze(c, f'clause_{i+1}')

          else:
              result = self.analyze(
                  c['text'],
                  c.get('id', f'clause_{i+1}')
              )

          results.append(result)

      return results

analyzer = ContractRiskAnalyzer()
print(' ContractRiskAnalyzer ready')

 ContractRiskAnalyzer ready


## 🔬 Run Sample Analyses

In [34]:
# ── Example 1: HIGH RISK clause ───────────────────────────────────────────────
clause_high = """
The contractor shall indemnify and hold harmless the Company from any and all claims.
The Company may terminate this agreement immediately and without cause at its sole discretion.
All intellectual property created shall be considered work for hire.
The contractor waives any right to arbitration or class action proceedings.
"""

result = analyzer.analyze(clause_high, 'high_risk_example')
print(json.dumps(result, indent=2))

{
  "clause_id": "high_risk_example",
  "original_text": "\nThe contractor shall indemnify and hold harmless the Company from any and all claims.\nThe Company may terminate this agreement immediately and without cause at its sole discretion.\nAll intellectual property created shall be considered work for hire.\nThe contractor waives any right to arbitration or class action proceedings.\n",
  "risk_level": "High",
  "risk_score": 46,
  "summary": "Significant legal risk. Do NOT sign without expert review. Detected 7 risk indicators with a total score of 46.",
  "flags": [
    {
      "flagged_term": "hold harmless",
      "risk_category": "Liability",
      "risk_weight": "8/10",
      "reason": "Waives right to sue \u2014 may eliminate critical legal remedies."
    },
    {
      "flagged_term": "without cause",
      "risk_category": "Termination",
      "risk_weight": "7/10",
      "reason": "Enables dismissal with no reason \u2014 highly unfavorable."
    },
    {
      "flagged_ter

In [35]:
# ── Example 2: MEDIUM RISK clause ────────────────────────────────────────────
clause_medium = """
Either party may terminate this agreement with a 30-day notice period.
The vendor shall keep all client data confidential and shall not disclose
any proprietary information to third parties without prior written consent.
Disputes shall be resolved by arbitration under the governing law of Delaware.
"""

result = analyzer.analyze(clause_medium, 'medium_risk_example')
print(json.dumps(result, indent=2))

{
  "clause_id": "medium_risk_example",
  "original_text": "\nEither party may terminate this agreement with a 30-day notice period.\nThe vendor shall keep all client data confidential and shall not disclose\nany proprietary information to third parties without prior written consent.\nDisputes shall be resolved by arbitration under the governing law of Delaware.\n",
  "risk_level": "High",
  "risk_score": 26,
  "summary": "Significant legal risk. Do NOT sign without expert review. Detected 6 risk indicators with a total score of 26.",
  "flags": [
    {
      "flagged_term": "arbitration",
      "risk_category": "Dispute Resolution",
      "risk_weight": "6/10",
      "reason": "Waives right to jury trial \u2014 limits appeal rights."
    },
    {
      "flagged_term": "terminate",
      "risk_category": "Termination",
      "risk_weight": "5/10",
      "reason": "Grants termination rights \u2014 risk depends on conditions."
    },
    {
      "flagged_term": "governing law",
      "ri

In [36]:
# ── Example 3: LOW RISK clause ────────────────────────────────────────────────
clause_low = """
The Company shall pay the contractor a monthly retainer of $5,000 on the first
business day of each month. All payments shall be made via bank transfer.
Both parties agree to communicate in good faith regarding any disagreements.
"""

result = analyzer.analyze(clause_low, 'low_risk_example')
print(json.dumps(result, indent=2))

{
  "clause_id": "low_risk_example",
  "original_text": "\nThe Company shall pay the contractor a monthly retainer of $5,000 on the first\nbusiness day of each month. All payments shall be made via bank transfer.\nBoth parties agree to communicate in good faith regarding any disagreements.\n",
  "risk_level": "Low",
  "risk_score": 0,
  "summary": "Minor risk indicators. Review recommended. Detected 0 risk indicators with a total score of 0.",
  "flags": [],
  "categories": {},
  "category_breakdown": {},
  "recommendations": []
}


In [37]:
# ── Example 4: Full contract (multiple clauses) ───────────────────────────────
contract = ["""SOFTWARE SERVICE AGREEMENT

This Software Service Agreement ("Agreement") is entered into between ABC Technologies ("Provider") and XYZ Enterprises ("Customer").

1. SERVICES

The Provider agrees to deliver software hosting, maintenance, and support services to the Customer for the duration of this Agreement.

2. TERM

This Agreement shall commence on the Effective Date and remain in effect for one year.

3. AUTOMATIC RENEWAL

This Agreement shall automatically renew for successive one-year periods unless the Customer provides written notice of termination at least ninety (90) days before the expiration date.

4. FEES AND PAYMENT

The Customer shall pay all fees within fifteen (15) days of invoice receipt. Any overdue payment shall incur a penalty of ten percent (10%) per month until paid in full.

5. TERMINATION

The Provider may terminate this Agreement at its sole discretion and without prior notice if it determines that continuation of services is no longer commercially viable.

6. LIMITATION OF LIABILITY

The Provider shall not be liable for any indirect, incidental, consequential, special, or punitive damages arising from the use of the services, even if advised of the possibility of such damages.

7. INDEMNIFICATION

The Customer agrees to indemnify, defend, and hold harmless the Provider, its officers, employees, and affiliates from and against any and all claims, damages, losses, liabilities, costs, and expenses arising from the Customer's use of the services.

8. CONFIDENTIALITY

Both parties shall maintain the confidentiality of proprietary information disclosed under this Agreement. Any breach of confidentiality by the Customer shall result in liquidated damages of $50,000 payable immediately upon written demand.

9. MODIFICATION OF TERMS

The Provider reserves the right to modify the terms of this Agreement at any time without obtaining prior consent from the Customer. Continued use of the services shall constitute acceptance of the modified terms.

10. DISPUTE RESOLUTION

Any dispute arising under this Agreement shall be resolved through binding arbitration conducted in the jurisdiction selected by the Provider.

11. FORCE MAJEURE

Neither party shall be liable for delays or failures caused by events beyond their reasonable control, including natural disasters, acts of government, or internet outages.

12. GOVERNING LAW

This Agreement shall be governed by and construed in accordance with the laws of the applicable jurisdiction.

13. ENTIRE AGREEMENT

This Agreement constitutes the entire understanding between the parties and supersedes all prior agreements, representations, and understandings."""]

full_result = analyzer.analyze_contract(contract)
print(json.dumps(full_result, indent=2))

[
  {
    "clause_id": "clause_1",
    "original_text": "SOFTWARE SERVICE AGREEMENT\n\nThis Software Service Agreement (\"Agreement\") is entered into between ABC Technologies (\"Provider\") and XYZ Enterprises (\"Customer\").\n\n1. SERVICES\n\nThe Provider agrees to deliver software hosting, maintenance, and support services to the Customer for the duration of this Agreement.\n\n2. TERM\n\nThis Agreement shall commence on the Effective Date and remain in effect for one year.\n\n3. AUTOMATIC RENEWAL\n\nThis Agreement shall automatically renew for successive one-year periods unless the Customer provides written notice of termination at least ninety (90) days before the expiration date.\n\n4. FEES AND PAYMENT\n\nThe Customer shall pay all fees within fifteen (15) days of invoice receipt. Any overdue payment shall incur a penalty of ten percent (10%) per month until paid in full.\n\n5. TERMINATION\n\nThe Provider may terminate this Agreement at its sole discretion and without prior notice

In [16]:
# ── Try your own clause ───────────────────────────────────────────────────────
my_clause = """Type your contract clause here and run this cell."""

my_result = analyzer.analyze(my_clause, 'my_clause')
print(json.dumps(my_result, indent=2))

{
  "clause_id": "my_clause",
  "original_text": "Type your contract clause here and run this cell.",
  "risk_level": "Low",
  "risk_score": 0,
  "summary": "Minor risk indicators. Review recommended. Score: 0.",
  "flags": [],
  "categories": {}
}
